# Milestone 3 lab, notebook 6 of 8: Same shape, new meaning

## Where we are

| Notebook | What you do |
|---|---|
| 1. A price has a start and an end | products. See one forgotten date count every sale twice |
| 2. Six rows, wrong everywhere | stores. Find out why the weakest check is the best one here |
| 3. A gap is not a zero | stock_counts. Look for a row that is not there |
| 4. The same person twice | members. Find one person behind two keys, and keep their data private |
| 5. A file you do not control | supplier_prices. A supplier renames a column, and nothing fails |
| **6. Same shape, new meaning** (you are here) | supplier_prices. Every number is valid, and 100 times too high |
| 7. A contract for the supplier | Write down what a file must look like, and check it at the door |
| 8. Cards side by side | Compare the seven cards. Then see what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import keep, lab_copy, money, on_lab, plant, prepare_week, production, sql

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 8 clean tables.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: An order for $209,558

Three days after the renamed column, **finance** opens the order list for Saturday 29 August. Compare the cost of the goods that come from the supplier `SUP-C` with the day before.

In [2]:
orders = sql("""
    SELECT r.for_date, count(*) AS rows, sum(r.order_qty) AS units_ordered,
           sum(r.order_cost_cents) AS order_cost_cents
    FROM output.replenishment_list r
    WHERE r.for_date IN (DATE '2026-08-28', DATE '2026-08-29')
      AND r.product_id IN (SELECT product_id FROM clean.supplier_prices WHERE supplier_id = 'SUP-C')
    GROUP BY r.for_date
    ORDER BY r.for_date
""")
orders["order_cost"] = orders["order_cost_cents"].map(money)
orders

,for_date,rows,units_ordered,order_cost_cents,order_cost
0,2026-08-28,90,321,99122,$991.22
1,2026-08-29,90,860,20955800,"$209,558.00"


$991.22 for Friday. $209,558.00 for Saturday.

The stores did order more for the weekend: 860 units, not 321. That is less than three times as much. The bill is more than 200 times as much. So the **cost of each unit** changed.

Here are four products, in `SUP-C`'s file of 21 August and its file of 28 August.

In [3]:
sql("""
    SELECT old.product_id, old.unit_cost AS in_the_file_of_21_august, new.unit_cost AS in_the_file_of_28_august
    FROM raw.supplier_prices old
    JOIN raw.supplier_prices new ON new.product_id = old.product_id
    WHERE old._file_name = 'supplier_prices_SUP-C_2026-08-21.csv'
      AND new._file_name = 'supplier_prices_SUP-C_2026-08-28.csv'
    ORDER BY old.product_id
    LIMIT 4
""")

,product_id,in_the_file_of_21_august,in_the_file_of_28_august
0,P-024,6.55,655
1,P-025,1.25,125
2,P-026,1.42,142
3,P-027,2.92,292


`6.55` became `655`. The supplier's program started to write the cost in **cents**. Lotus reads the column as dollars, as it always did. So olive oil that costs $6.55 is now $655.00 a bottle.

## Step 2: Every check you have is green

Go through what you built. First, the schema check from the last notebook.

In [4]:
schema = production("files_have_their_columns")
print("Findings of the schema check for this file:", len(schema[schema["thing"] == "supplier_prices_SUP-C_2026-08-28.csv"]))

filled = production("cost_is_filled")
print("Rows of SUP-C with no cost:                ", len(filled[filled["supplier_id"] == "SUP-C"]))

Findings of the schema check for this file: 0
Rows of SUP-C with no cost:                 0


In [5]:
sql("""
    SELECT count(*) AS rows, min(unit_cost_cents) AS smallest_cost_cents, max(unit_cost_cents) AS largest_cost_cents,
           count(*) - count(unit_cost_cents) AS empty
    FROM clean.supplier_prices
    WHERE supplier_id = 'SUP-C' AND price_date = DATE '2026-08-28'
""")

,rows,smallest_cost_cents,largest_cost_cents,empty
0,15,4000,65500,0


Every column is there, with the right name. Every cell is filled. Every value is a number above zero.

The shape of the file is perfect. Its **meaning** changed. The same column that meant dollars now means cents.

This is called a **change of meaning**: the shape stays the same and the meaning changes. No schema check can see it, because there is nothing wrong with the schema.

| | Schema drift (notebook 5) | A change of meaning |
|---|---|---|
| What changed | A column name | What a number stands for |
| The schema check | Fails | Passes |
| The not-null check | Fails | Passes |
| What reaches finance | No cost | A cost 100 times too high |

The second one is worse. An empty cost makes people ask. A wrong cost gets paid.

## Step 3: First way to catch it. Against the last file

A table cannot be asked "is this number true?". But this supplier sent a file last week. Costs move a little from week to week. They do not jump 100 times.

Join each cost to the same product in the file one week before.

In [6]:
COST_NEAR_THE_LAST_FILE = """
    SELECT new.supplier_id || ' ' || new.product_id AS thing, new.price_date AS day,
           new.supplier_id, new.product_id,
           old.unit_cost_cents AS last_cost_cents, new.unit_cost_cents AS new_cost_cents
    FROM clean.supplier_prices new
    JOIN clean.supplier_prices old
      ON old.supplier_id = new.supplier_id AND old.product_id = new.product_id
     AND old.price_date = new.price_date - 7
    WHERE new.unit_cost_cents > 2 * old.unit_cost_cents
       OR new.unit_cost_cents * 2 < old.unit_cost_cents
"""
yours = sql(COST_NEAR_THE_LAST_FILE)
print(len(yours), "costs are far from the last file")
yours.head(4)

15 costs are far from the last file


,thing,day,supplier_id,product_id,last_cost_cents,new_cost_cents
0,SUP-C P-024,2026-08-28,SUP-C,P-024,655,65500
1,SUP-C P-025,2026-08-28,SUP-C,P-025,125,12500
2,SUP-C P-026,2026-08-28,SUP-C,P-026,142,14200
3,SUP-C P-027,2026-08-28,SUP-C,P-027,292,29200


In [7]:
keep("cost_near_the_last_file", yours)

Your query:           15 bad rows
The production check: 15 bad rows  (A cost is near the cost in the last file)
The same rows.


Fifteen rows: every product of `SUP-C`, on 28 August, each exactly 100 times its last cost.

This is the baseline idea from milestone 2, on a new table. There the baseline was a normal day of sales. Here it is the supplier's own last file.

The limit is "at most twice, at least half". Is that too tight? Costs do really change. Here is every real change in the five weeks.

In [8]:
sql("""
    SELECT new.supplier_id, new.product_id, new.price_date,
           old.unit_cost_cents AS last_cost_cents, new.unit_cost_cents AS new_cost_cents,
           round(100.0 * new.unit_cost_cents / old.unit_cost_cents) AS percent_of_last
    FROM clean.supplier_prices new
    JOIN clean.supplier_prices old
      ON old.supplier_id = new.supplier_id AND old.product_id = new.product_id
     AND old.price_date = new.price_date - 7
    WHERE new.unit_cost_cents <> old.unit_cost_cents AND new.price_date < DATE '2026-08-28'
    ORDER BY new.price_date
""")

,supplier_id,product_id,price_date,last_cost_cents,new_cost_cents,percent_of_last
0,SUP-A,P-005,2026-08-10,247,252,102
1,SUP-B,P-021,2026-08-11,447,467,104
2,SUP-C,P-034,2026-08-14,227,223,98


Three real changes, each of a few cents. None is near the limit, and the check returns none of them. A check that fired on every real price change would be switched off within a week.

## Step 4: Second way. Against the shelf price

There is another fact you can lean on, and it needs no history. A shop sells a product for more than it pays for it.

Join each cost to the product's shelf price on that day.

In [9]:
COST_BELOW_THE_SHELF_PRICE = """
    SELECT s.supplier_id || ' ' || s.product_id AS thing, s.price_date AS day,
           s.supplier_id, s.product_id, p.product_name,
           s.unit_cost_cents, p.unit_price_cents AS shelf_price_cents
    FROM clean.supplier_prices s
    JOIN clean.products p
      ON p.product_id = s.product_id
     AND s.price_date BETWEEN p.valid_from AND coalesce(p.valid_to, DATE '9999-12-31')
    WHERE s.unit_cost_cents >= p.unit_price_cents
"""
yours = sql(COST_BELOW_THE_SHELF_PRICE)
print(len(yours), "products cost more than they sell for")
yours.head(4)

15 products cost more than they sell for


,thing,day,supplier_id,product_id,product_name,unit_cost_cents,shelf_price_cents
0,SUP-C P-024,2026-08-28,SUP-C,P-024,Olive oil 500ml,65500,949
1,SUP-C P-025,2026-08-28,SUP-C,P-025,Pasta 500g,12500,189
2,SUP-C P-026,2026-08-28,SUP-C,P-026,Tomato sauce,14200,259
3,SUP-C P-027,2026-08-28,SUP-C,P-027,Basmati rice 1kg,29200,429


In [10]:
keep("cost_below_the_shelf_price", yours)

Your query:           15 bad rows
The production check: 15 bad rows  (A product costs less than it sells for)
The same rows.


The same fifteen rows.

Two checks on one fault. Are they the same check twice? Plant a different fault and see. In a private copy, one cost drops to a tenth: a decimal point slipped the other way.

In [11]:
lab_copy("clean.supplier_prices")
plant("""
    UPDATE lab.supplier_prices SET unit_cost_cents = unit_cost_cents / 10
    WHERE product_id = 'P-024' AND price_date = DATE '2026-08-21'
""")
for key in ("cost_near_the_last_file", "cost_below_the_shelf_price"):
    found = production(key, on_the_lab=True)
    here = found[(found["product_id"] == "P-024") & (found["day"].astype(str) == "2026-08-21")]
    print(f"{key:<28} {'catches it' if len(here) else 'is silent'}")

lab.supplier_prices is a fresh copy of clean.supplier_prices: 198 rows.
Planted in the lab copy.
cost_near_the_last_file      catches it
cost_below_the_shelf_price   is silent


The first catches it. The second is silent: a cost that is too **low** is still below the shelf price.

So they are two different checks. One knows the supplier's history. The other knows a rule of the business. Each sees faults the other cannot.

## Step 5: The third way

Both checks speak **after** the load. The wrong costs are already in the warehouse, and already on the order list.

There is a place to catch it earlier: at the door, before the file is loaded. For that, Lotus and the supplier must first write down what `unit_cost` means. That is the next notebook.

## What you now know

- A **change of meaning** keeps the shape and changes what a value stands for, such as dollars sent as cents. No schema check can see it.
- It is worse than a missing column: a wrong number gets used.
- You catch it by comparing with something outside the row: the sender's last file, or another table.
- Prove the limit on real changes too. Three real cost changes stayed silent.

**Say it in an interview:** "A supplier started sending costs in cents where we had agreed dollars. Same columns, same types, every value a valid number, and every cost 100 times too high. A schema check passes that. I caught it two ways: each cost against the same product in the supplier's last file, and each cost against our shelf price. They are different checks: a cost that drops to a tenth fails only the first."

**The theory:** [Schema Evolution & Data Contracts](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/schema-evolution-and-data-contracts), section "Same shape, new meaning".

Next: [notebook 7](07_a_contract_for_the_supplier.ipynb). Write the agreement down.